---
title: "Private Estimation — Self-study notebook"
subtitle: "Applied Data Privacy"
format:
  html:
    page-layout: full
    toc: false
    toc-depth: 2
    code-tools: true
    code-fold: true
    code-overflow: wrap
    include-in-header:
      text: |
        <style>
        .cell-output-display img, .cell-output-display .plotly-graph-div { max-width: 100%; height: auto; }
        </style>
execute:
  enabled: true
  warning: false
  message: false
jupyter: python3
---

This is the **self-study** companion to the lecture deck: full narrative + all code, meant to be
read and run. For the slide version (code hidden), open the [presentation deck](../../lecture-presentations/private-estimation/presentation.html).


In [ ]:
#| output: false
#| echo: false
# Environment setup: pick up local edits to libdpy/ without restarting the kernel,
# and install the public library when missing (e.g. on Google Colab).
try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except Exception:
    pass

try:
    import libdpy
except ImportError:
    %pip install -q "libdpy[notebook] @ git+https://github.com/applied-dp-course/pub_lib.git"
    import libdpy

%matplotlib inline


In [ ]:
# Environment setup: pick up local edits to libdpy/ without restarting the kernel.
try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except Exception:
    pass

try:
    import libdpy
except ImportError:
    %pip install -q "libdpy[notebook] @ git+https://github.com/applied-dp-course/pub_lib.git"
    import libdpy


In [ ]:
from functools import partial

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.io as pio

from libdpy.assignment_specific.private_estimation.lecture_figures import (
    build_median_section_artifact,
    build_raw_mean_section_artifact,
    make_accuracy_leaderboard_figure,
    make_oracle_income_histogram_figure,
)
from libdpy.assignment_specific.private_estimation.utils import (
    PUBLIC_INCOME_BIN_EDGES,
    PUBLIC_INCOME_CANDIDATES,
    PUBLIC_INVERSE_SENSITIVITY_THRESHOLDS,
    PUBLIC_QUANTILE_LOWER,
    PUBLIC_QUANTILE_UPPER,
    PUBLIC_SCALE_DIFF_BIN_EDGES,
    DEFAULT_K_SIGMA,
    DataProvenance,
    empirical_quantile_clipped_mean,
    empirical_quantile_output_law,
    empirical_mu_sigma_clipped_mean,
    accuracy_summary_table,
    audit_panel,
    budget_ledger_table,
    empirical_mu_sigma_bounds,
    empirical_mu_sigma_clipped_stat,
    empirical_mu_sigma_noise_scale,
    empirical_mu_sigma_output_law,
    evaluate_accuracy,
    extract_income,
    fixed_bin_noisy_histogram,
    gaussian_histogram_mean,
    clipped_noisy_mean,
    gaussian_noise_std,
    histogram_replacement_trace,
    private_mu_sigma_clipped_mean,
    private_scale_from_pairwise_diffs,
    enforce_clip_bounds,
    private_inverse_sensitivity_quantile,
    private_quantile_exponential,
    public_range_noisy_mean,
    rank_utility,
    load_fulton,
    median_value_plus_noise,
    median_perturbation_trace,
    neighbor_witness_table,
    proposals_diagnoses_repairs_from_leaderboard,
    quantile_clipping_perturbation_trace,
    raw_mean_perturbation_trace,
    build_engineered_quantile_neighbor_pair,
    replace_one_row,
    select_typical_median_pair,
    replacement_swap_influences,
    population_rank_cdf,
    select_typical_mu_sigma_pair,
    split_base_and_pool,
    stability_summary,
)
from libdpy.assignment_specific.private_estimation.embed_interactives import (
    PrivateEstimationAuditROCVisualizer,
)
from libdpy.visualization.roc_plots import TheoryROCVisualizer
from libdpy.assignment_specific.private_estimation.visualizations import (
    PROVENANCE_STYLE,
    sorted_dataset_bars_figure,
    sorted_neighbor_bars_figure,
)

# PUBLIC-MENU CONTRACT: all candidate grids, bin edges, transformations,
# hypothesis lists, clipping multipliers, and subgroup definitions are fixed
# before any private sample is touched, or come from designated public data.
SEED = 42
N = 1000
EPS_TOTAL = 1.0
DELTA = 1e-2

# Accuracy-regime contract. Many independent datasets, one mechanism draw each,
# so signed-error histograms look smooth without repeated calls per dataset.
N_DATASETS_RECOMMENDED = 5_000
N_RUNS_RECOMMENDED = 1
N_DATASETS = N_DATASETS_RECOMMENDED
N_RUNS = N_RUNS_RECOMMENDED
HISTOGRAM_TRIALS = 5_000

assert N == 1000
assert EPS_TOTAL == 1.0

# Epsilon budget: name slices and assert sums (even when the whole budget goes to one step).
EPS_MEAN = EPS_TOTAL
assert abs(EPS_MEAN - EPS_TOTAL) < 1e-12

# Capstone Gaussian-localization split (S8).
EPS_SCALE = 0.25
EPS_LOCATION = 0.25
EPS_GAUSS_MEAN = 0.50
assert abs((EPS_SCALE + EPS_LOCATION + EPS_GAUSS_MEAN) - EPS_TOTAL) < 1e-12

# Child seeds (documented map from lecture root SEED).
SEED_MAP = {
    "accuracy_public_range_mean": SEED,
    "typical_mu_sigma_pair": SEED + 11,
    "accuracy_private_ms_one": SEED + 2,
    "accuracy_private_ms_iter": SEED + 3,
    "accuracy_median_value_noise": SEED,
    "accuracy_private_quantile": SEED + 6,
    "accuracy_private_qcm": SEED + 7,
    "accuracy_gaussian_loc_synth": SEED + 9,
    "accuracy_gaussian_loc_fulton": SEED + 8,
    "audit_ms_repair_one": SEED + 40,
    "audit_ms_repair_three": SEED + 41,
    "typical_median_pair": SEED + 17,
    "accuracy_empirical_quantile": SEED + 19,
    "audit_s3_empirical_quantile": SEED + 18,
    "gaussian_synth_population": SEED + 100,
}


df = load_fulton()
income_all = extract_income(df)
D, witness_pool = split_base_and_pool(income_all, n_base=N, seed=SEED)
rng = np.random.default_rng(SEED)
# Keep Plotly figures interactive in notebooks when the frontend supports MIME rendering.
pio.renderers.default = "plotly_mimetype+notebook_connected"
leaderboard_parts = []

mean_targets = {'value': lambda x: float(np.mean(x))}
median_targets = {
    'value': lambda x: float(np.median(x)),
    'rank': lambda released: population_rank_cdf(income_all, released),
}

print(f"Fulton rows: {len(df):,} | accuracy sample N={N:,} | substitution adjacency")
print(f"Main privacy contract: epsilon_total={EPS_TOTAL:.1f}")
print(f"Accuracy sweep in this notebook: {N_DATASETS} datasets x {N_RUNS} run (recommended: {N_DATASETS_RECOMMENDED} x {N_RUNS_RECOMMENDED})")
print(f"Empirical audit histograms: {HISTOGRAM_TRIALS} mechanism draws per neighboring dataset")


# Lecture 5 — Private Estimation by Auditing Failed Estimators

**Thesis:** private estimation is not "compute the statistic, then add noise." Every data-dependent choice made before the noise is also part of the release.

**Running problem:** estimate a one-number income statistic from a private Fulton-style sample of **N = 1000** with total privacy budget **epsilon = 1.0**. We start with the mean, then move through scale-based clipping, quantile-based clipping, medians, and histogram localization.


## S0 — Contract and Method

Every section follows the same order: visible mechanism code, an accuracy comparison under the fixed contract when appropriate, then a separate witness/audit explanation if the method is invalid. The order is deliberately cumulative: mean estimation first, then empirical `mu ± k sigma`, empirical quantiles, median/inverse sensitivity, private quantile clipping, and finally noisy histograms for binning and Gaussian-style localization.


Before proposing mechanisms, look once at the full Fulton income dataset as a non-private teaching artifact. The next plot uses the same sorted-dataset visual language as the neighboring-pair examples later in the notebook. X-axis: sorted row index in the full dataset. Y-axis: income.


In [ ]:
fig = sorted_dataset_bars_figure(
    income_all,
    title=f'Full Fulton income dataset (n={len(income_all):,}) — exact, not a private release',
    provenance=DataProvenance.ORACLE_DIAGNOSTIC,
    yscale='linear',
    show_quantiles=True,
    show_median=True,
)
plt.show()


The same full dataset is shown once as an exact histogram. This is the tempting non-private EDA view that later motivates fixed public bins and noisy histograms. X-axis: public income bins in dollars. Y-axis: exact row counts.


In [ ]:
fig = make_oracle_income_histogram_figure(seed=SEED, log_scale=False)
plt.show()


## S1 — Public-Range Noisy Mean

First proposal: release the sample mean with Gaussian noise calibrated to a **public salary-domain range**, not to the observed Fulton range. This is a valid bounded-domain mean-query template only after choosing the appropriate Gaussian calibration constants; here the point is utility: if the public range is one or two orders of magnitude larger than the real salaries, the required noise is correspondingly large.


In [ ]:
RAW_MEAN_PUBLIC_RANGE = 10_000_000.0
raw_mean_noise_std = gaussian_noise_std(RAW_MEAN_PUBLIC_RANGE / N, EPS_MEAN, DELTA)
observed_fulton_range = float(np.max(income_all) - np.min(income_all))
raw_mean_attempt = partial(
    public_range_noisy_mean,
    public_range=RAW_MEAN_PUBLIC_RANGE,
    epsilon=EPS_MEAN,
    delta=DELTA,
)

print(f'Epsilon: {EPS_MEAN}')
print(f'Public salary-domain range: {RAW_MEAN_PUBLIC_RANGE:,.0f}')
print(f'Observed full-Fulton income range, for context only: {observed_fulton_range:,.0f}')
print(f'Gaussian noise std: {raw_mean_noise_std:,.1f}')


One accuracy draw before the sweep: draw a size-`N` dataset from Fulton, run the mechanism once, and compare the release to the same sample's non-private mean.


In [ ]:
_acc_rng = np.random.default_rng(SEED_MAP["accuracy_public_range_mean"])
_one_draw = _acc_rng.choice(income_all, size=N, replace=False)
_one_run_rng = np.random.default_rng(_acc_rng.integers(0, 2**31 - 1))
_one_estimate = raw_mean_attempt(_one_draw, rng=_one_run_rng)
_one_target = float(np.mean(_one_draw))
_one_error = _one_estimate - _one_target
print(
    f"single draw: target={_one_target:,.0f}, estimate={_one_estimate:,.0f}, "
    f"error={_one_error:,.0f}"
)


The next code cell displays the signed-error decomposition for this public-range noisy mean on ordinary size-`N=1000` accuracy samples. The three lines are `sample mean - full Fulton mean`, `released estimate - sample mean`, and `released estimate - full Fulton mean`. X-axis: signed error in dollars. Y-axis: probability per bin.


In [ ]:
raw_accuracy_rows = evaluate_accuracy(
    raw_mean_attempt,
    income_all,
    mean_targets,
    n=N,
    n_datasets=N_DATASETS,
    n_runs=N_RUNS,
    seed=SEED_MAP["accuracy_public_range_mean"],
    method='public-range noisy mean',
    privacy_status='valid',
    estimate_target='mean',
    epsilon_total=EPS_TOTAL,
    notes='epsilon=1 noise calibrated to a conservative public salary-domain range',
)
leaderboard_parts.append(raw_accuracy_rows)
fig = make_accuracy_leaderboard_figure(raw_accuracy_rows, estimate_target='mean', error_metric='value')
plt.show()


This cell computes a typical-data substitution probe for the raw mean without rendering dataframe output. It is a non-audit diagnostic used to support the surrounding discussion.


In [ ]:
typical_swaps = replacement_swap_influences(
    D, witness_pool, lambda x: float(np.mean(x)), n_swaps=300, seed=SEED
)
typical_swaps_summary = stability_summary(
    typical_swaps, 'typical substitution swaps for raw mean'
)


This cell constructs the extreme-real substitution diagnostic for the public-range raw mean without rendering witness dataframes. The point is referenced in prose; the public-range mechanism is not treated as broken by this row.


In [ ]:
raw_mean_artifact = build_raw_mean_section_artifact(n_base=N, seed=SEED)
raw_mean_artifact = raw_mean_artifact.__class__(
    D=raw_mean_artifact.D,
    D_prime=raw_mean_artifact.D_prime,
    out_idx=raw_mean_artifact.out_idx,
    witness=raw_mean_artifact.witness,
    mean_D=raw_mean_artifact.mean_D,
    mean_D_prime=raw_mean_artifact.mean_D_prime,
    noise_scale=raw_mean_noise_std,
    epsilon=EPS_TOTAL,
)
raw_mean_witness_rows = neighbor_witness_table(
    section='S1 public-range noisy mean',
    d_description=f'Fulton sample (n={len(raw_mean_artifact.D)})',
    d_prime_description=(
        f'substitute row {raw_mean_artifact.out_idx} '
        f'with genuine high earner {raw_mean_artifact.witness:,.0f}'
    ),
    adjacency='substitution (one row out, one row in; n unchanged)',
    n=len(raw_mean_artifact.D),
    algorithm_effect='mean shift is small relative to public-range Gaussian std',
    audit_target='released noisy mean',
    provenance=DataProvenance.EXTREME_REAL,
)
raw_mean_trace_rows = raw_mean_perturbation_trace(
    raw_mean_artifact.D,
    raw_mean_artifact.D_prime,
    raw_mean_artifact.out_idx,
    raw_mean_artifact.noise_scale,
)


**Diagnosis:** a raw mean can be made private by assuming a very large public data-domain range, but the resulting noise is too large to be useful. The next attempt tries to reduce noise by learning a narrower `mu ± k sigma` range; that learning step is where privacy can fail.


## S2 — Empirical `mu ± k sigma` Clipping

Bad proposal: learn `mu` and `sigma` from the private sample, clip to `mu ± k sigma`, then add epsilon = 1 Gaussian noise to the clipped mean. This often looks good on ordinary data because the empirical mean and standard deviation barely move under typical row swaps. It is still invalid: the clipping interval and the noise standard deviation are data-dependent private releases.


In [ ]:
K_SIGMA = DEFAULT_K_SIGMA
empirical_mu_sigma_clipping_attempt = partial(
    empirical_mu_sigma_clipped_mean,
    epsilon=EPS_TOTAL,
    k=K_SIGMA,
)


First look at an ordinary substitution in a typical size-`N=1000` Fulton sample. This is a stability probe, not a privacy proof: it constructs a neighboring pair whose learned `mu ± k sigma` interval barely moves, explaining why the invalid rule is tempting on ordinary data.


In [ ]:
typical_ms_shift, typical_ms_out_idx, typical_ms_in_value, typical_ms_D_prime = select_typical_mu_sigma_pair(
    D, witness_pool, k=K_SIGMA, seed=SEED_MAP["typical_mu_sigma_pair"]
)
typical_ms_noise = max(
    empirical_mu_sigma_noise_scale(D, EPS_TOTAL, k=K_SIGMA),
    empirical_mu_sigma_noise_scale(typical_ms_D_prime, EPS_TOTAL, k=K_SIGMA),
)


The next plot shows the typical neighboring pair. The sorted samples and learned upper clipping lines are almost unchanged, which explains why this invalid proposal is tempting on ordinary data.


In [ ]:
fig = sorted_neighbor_bars_figure(
    D,
    typical_ms_D_prime,
    title='Typical pair: empirical mu±4 sigma barely moves',
    provenance=DataProvenance.REAL_DATA,
    show_quantiles=False,
    mu_sigma_bounds_fn=lambda values: empirical_mu_sigma_bounds(values, k=K_SIGMA),
    mu_sigma_k=K_SIGMA,
)
plt.show()


The next plot shows the corresponding analytical Gaussian ROC for the same typical pair. X-axis: false positive rate. Y-axis: true positive rate. The two output laws use the deterministic clipped means and the **common Gaussian noise std required for epsilon = 1 at `DELTA` on this pair**, not the broken proposal's data-dependent std. This calibrated analytical ROC shows the epsilon-1 bound line; it does not mark a selected-threshold circle because no audit threshold is being selected in this plot.


In [ ]:
typical_ms_loc, _ = empirical_mu_sigma_output_law(D, EPS_TOTAL, k=K_SIGMA)
typical_ms_loc_prime, _ = empirical_mu_sigma_output_law(typical_ms_D_prime, EPS_TOTAL, k=K_SIGMA)
typical_ms_required_std = gaussian_noise_std(abs(typical_ms_loc - typical_ms_loc_prime), EPS_TOTAL, DELTA)
TheoryROCVisualizer(
    distribution='Gaussian',
    scale=10.495637332790803,
    delta=1e-06,
    selectable_distribution=False,
    loc_negative=32679.877137570682,
    scale_negative=10.495637332790803,
    loc_positive=32682.36150090521,
    scale_positive=10.495637332790803,
    bound_epsilon=1.0,
    show_governing_point=False,
).embed()


Now move to an engineered size-`N=1000` leverage example. The pair is still a substitution-neighbor pair with the same size on both sides, but the incoming value is fabricated to make the empirical standard deviation and learned clipping interval move visibly.


In [ ]:
engineered_ms_D = D.copy()
engineered_ms_out_idx = int(np.argmin(engineered_ms_D))
engineered_ms_witness = 2_000_000.0
engineered_ms_D_prime = replace_one_row(
    engineered_ms_D, engineered_ms_out_idx, engineered_ms_witness
)


The next plot shows the engineered neighboring pair. Only **one row** differs between `D` and `D'`. The horizontal lines mark the sample **mean μ** and the learned upper clip **μ+4σ** (with empirical **σ** in the legend). Substituting one upper-tail row barely moves μ, but it inflates σ and therefore moves the clipping bound.


In [ ]:
fig = sorted_neighbor_bars_figure(
    engineered_ms_D,
    engineered_ms_D_prime,
    title='Engineered pair: one size-1000 substitution moves empirical σ (μ±4σ)',
    provenance=DataProvenance.CONSTRUCTED_WITNESS,
    yscale='linear',
    show_quantiles=False,
    mu_sigma_bounds_fn=lambda values: empirical_mu_sigma_bounds(values, k=K_SIGMA),
    mu_sigma_k=K_SIGMA,
)
plt.show()


This is the primary privacy-failure ROC for the engineered size-`N=1000` pair. It uses the analytical output laws of the broken data-dependent `mu ± k sigma` proposal: two Gaussians with different deterministic clipped means and different learned noise stds on `D` and `D'`. The thick purple line is the **tight** `(ε, δ)`-DP bound for this ROC; the open circle marks the governing point where it meets the curve. That tight ε exceeds the claimed `epsilon=1` budget.


In [ ]:
engineered_ms_loc, engineered_ms_scale = empirical_mu_sigma_output_law(
    engineered_ms_D, EPS_TOTAL, k=K_SIGMA
)
engineered_ms_loc_prime, engineered_ms_scale_prime = empirical_mu_sigma_output_law(
    engineered_ms_D_prime, EPS_TOTAL, k=K_SIGMA
)
TheoryROCVisualizer(
    distribution='Gaussian',
    scale=838.3707143443063,
    delta=1e-06,
    selectable_distribution=False,
    loc_negative=32679.877137570682,
    scale_negative=838.3707143443063,
    loc_positive=34646.1024362846,
    scale_positive=1254.4000868708986,
    show_governing_point=True,
    compute_epsilon=True,
    show_compute_epsilon_toggle=False,
).embed()


The private repair spends epsilon on noisy truncated-moment localization and the final clipped mean. The budget ledger is computed for bookkeeping but not rendered as dataframe output in the mainline notebook.


In [ ]:
PUBLIC_MS_LOWER = 0.0
PUBLIC_MS_UPPER = float(PUBLIC_INCOME_CANDIDATES[-1])
EPS_MS_LOCALIZE = 0.60
EPS_MS_MEAN = 0.40
MIN_PRIVATE_SIGMA = 2_500.0
assert abs((EPS_MS_LOCALIZE + EPS_MS_MEAN) - EPS_TOTAL) < 1e-12

private_ms_one_mechanism = lambda x, rng: private_mu_sigma_clipped_mean(
    x,
    rng,
    eps_localize=EPS_MS_LOCALIZE,
    eps_mean=EPS_MS_MEAN,
    n_rounds=1,
    k=K_SIGMA,
    public_lower=PUBLIC_MS_LOWER,
    public_upper=PUBLIC_MS_UPPER,
    min_private_sigma=MIN_PRIVATE_SIGMA,
)
private_ms_three_mechanism = lambda x, rng: private_mu_sigma_clipped_mean(
    x,
    rng,
    eps_localize=EPS_MS_LOCALIZE,
    eps_mean=EPS_MS_MEAN,
    n_rounds=3,
    k=K_SIGMA,
    public_lower=PUBLIC_MS_LOWER,
    public_upper=PUBLIC_MS_UPPER,
    min_private_sigma=MIN_PRIVATE_SIGMA,
)

private_ms_one = private_ms_one_mechanism(D, rng)
private_ms_iter = private_ms_three_mechanism(D, rng)
private_ms_budget_rows = budget_ledger_table(private_ms_iter)


Before looking at accuracy, audit the valid private repair on the same engineered size-`N=1000` neighboring pair. Each figure matches the auditing lecture: empirical output densities for `H_0` and `H_1` with the ROC-selected threshold, plus the finite-sample empirical ROC with the governing point and the claimed `epsilon = 1` bound.

The extractor audits only the final released mean (`result['estimate']`), so plug-in ε̂ is an empirical **lower bound** on that scalar release — consistent with, but not a proof of, the composed ε=1 claim. It cannot see the private μ̂/σ̂ localization that consumed 60% of the budget.


In [ ]:
MU_SIGMA_REPAIR_AUDIT_TRIALS = HISTOGRAM_TRIALS
panel_ms_repair_one = audit_panel(
    private_ms_one_mechanism,
    engineered_ms_D,
    engineered_ms_D_prime,
    n_trials=MU_SIGMA_REPAIR_AUDIT_TRIALS,
    delta=DELTA,
    seed=SEED_MAP["audit_ms_repair_one"],
    extractor=lambda result: result['estimate'],
    claimed_eps=EPS_TOTAL,
    adversary_statistic='released private mu±4 sigma estimate',
)
eps_plug_ms_one = panel_ms_repair_one.eps_plug
print(
    f"tau*={panel_ms_repair_one.tau_star:.3g}, eps_plug={eps_plug_ms_one:.3g}, "
    f"separates={panel_ms_repair_one.separates}"
)
PrivateEstimationAuditROCVisualizer(scene="ms-repair-one").embed()


Repeat the same audit on the three-round localization repair with the same total `epsilon = 1` budget. Plug-in ε̂ values for one and three rounds are typically similar (both well below the claimed budget): iteration sharpens clip bounds for **accuracy**, not privacy.

In [ ]:
panel_ms_repair_three = audit_panel(
    private_ms_three_mechanism,
    engineered_ms_D,
    engineered_ms_D_prime,
    n_trials=MU_SIGMA_REPAIR_AUDIT_TRIALS,
    delta=DELTA,
    seed=SEED_MAP["audit_ms_repair_three"],
    extractor=lambda result: result['estimate'],
    claimed_eps=EPS_TOTAL,
    adversary_statistic='released private mu±4 sigma estimate',
)
eps_plug_ms_three = panel_ms_repair_three.eps_plug
print(
    f"tau*={panel_ms_repair_three.tau_star:.3g}, eps_plug={eps_plug_ms_three:.3g}, "
    f"separates={panel_ms_repair_three.separates}"
)
PrivateEstimationAuditROCVisualizer(scene="ms-repair-three").embed()


The next figure compares the one-step and multi-step private `mu, sigma` repairs on the mean track. Both spend the same total ε; extra localization rounds split that budget across rounds to **narrow the final clipping interval** and improve accuracy — they do not change the privacy guarantee.


One accuracy draw for the one-step private mu±sigma repair before the sweep.


In [ ]:
_pms_rng = np.random.default_rng(SEED_MAP["accuracy_private_ms_one"])
_pms_draw = _pms_rng.choice(income_all, size=N, replace=False)
_pms_run_rng = np.random.default_rng(_pms_rng.integers(0, 2**31 - 1))
_pms_result = private_ms_one_mechanism(_pms_draw, _pms_run_rng)
_pms_target = float(np.mean(_pms_draw))
print(
    f"single draw: target={_pms_target:,.0f}, estimate={_pms_result['estimate']:,.0f}, "
    f"error={_pms_result['estimate'] - _pms_target:,.0f}"
)


Typical-accuracy sweep for the one-step private mu±sigma repair (`N_DATASETS` replicates × `N_RUNS` mechanism draw each). X-axis: signed error in dollars. Y-axis: probability per bin.


In [ ]:
private_ms_one_rows = evaluate_accuracy(
    private_ms_one_mechanism,
    income_all,
    mean_targets,
    n=N,
    n_datasets=N_DATASETS,
    n_runs=N_RUNS,
    seed=SEED_MAP["accuracy_private_ms_one"],
    method='private mu-sigma clipped mean (1 step)',
    privacy_status='valid, empirical audit only',
    estimate_target='mean',
    epsilon_total=EPS_TOTAL,
    notes='private truncated moments plus noisy clipped mean',
)
private_ms_iter_rows = evaluate_accuracy(
    private_ms_three_mechanism,
    income_all,
    mean_targets,
    n=N,
    n_datasets=N_DATASETS,
    n_runs=N_RUNS,
    seed=SEED_MAP["accuracy_private_ms_iter"],
    method='private mu-sigma clipped mean (3 steps)',
    privacy_status='valid, empirical audit only',
    estimate_target='mean',
    epsilon_total=EPS_TOTAL,
    notes='iterative private truncated moments plus noisy clipped mean',
)
leaderboard_parts.append(pd.concat([private_ms_one_rows, private_ms_iter_rows], ignore_index=True))
fig = make_accuracy_leaderboard_figure(
    pd.concat([private_ms_one_rows, private_ms_iter_rows], ignore_index=True),
    estimate_target='mean',
    error_metric='value',
)
plt.show()



**Diagnosis:** empirical center and scale are private-data releases. A private approximation can be built from truncated noisy moments and iterated with budget splitting, but its useful accuracy is an empirical question rather than something certified by the failed empirical sensitivity argument.


## S3 — Empirical Quantile Clipping

Bad proposal: learn the 1% and 99% bounds from the private sample, clip to those bounds, then add Gaussian noise to the clipped mean. This is the quantile analogue of the empirical `mu ± k sigma` bug: the final noise standard deviation is calibrated to a learned interval, but the interval and therefore the variance are themselves data-dependent private releases.


In [ ]:
LOW_Q, HIGH_Q = 0.01, 0.99
empirical_quantile_clipping_attempt = partial(
    empirical_quantile_clipped_mean,
    epsilon=EPS_TOTAL,
    low_q=LOW_Q,
    high_q=HIGH_Q,
)


The broken empirical-quantile proposal is diagnosed with substitution pairs and ROCs rather than an accuracy histogram. This follows the valid-only accuracy-display rule: invalid methods can look accurate, but the section-level histogram is reserved for repaired private methods.


First look at a typical size-`N=1000` pair. We replace one row by a huge value, but the value is clipped near the empirical 99% quantile, so both the clipped mean and the Gaussian std barely move. This is why the proposal is tempting on benign data.


In [ ]:
HUGE_SUBSTITUTION = 200_000.0
typical_quantile_D = D.copy()
typical_quantile_out_idx = int(np.argmin(D))
typical_quantile_D_prime = replace_one_row(
    typical_quantile_D, typical_quantile_out_idx, HUGE_SUBSTITUTION
)


The next plot shows a typical neighboring pair. Even a huge substituted row is clipped near the empirical 99% quantile, so the two sorted datasets are almost indistinguishable after clipping.


In [ ]:
fig = sorted_neighbor_bars_figure(
    typical_quantile_D,
    typical_quantile_D_prime,
    title='Typical pair: one huge substituted row is clipped away',
    yscale='linear',
    provenance=DataProvenance.REAL_DATA,
)
plt.show()


The next plot shows the analytical Gaussian ROC for that typical pair. It uses the deterministic clipped means and the **common Gaussian std required for epsilon = 1 at `DELTA` on this pair**, not the broken proposal's learned std. This calibrated analytical ROC shows the epsilon-1 bound line without a selected-threshold circle.


In [ ]:
typical_mu, _ = empirical_quantile_output_law(typical_quantile_D, LOW_Q, HIGH_Q, EPS_TOTAL, DELTA)
typical_mu_prime, _ = empirical_quantile_output_law(typical_quantile_D_prime, LOW_Q, HIGH_Q, EPS_TOTAL, DELTA)
typical_scale = gaussian_noise_std(
    typical_mu - typical_mu_prime, EPS_TOTAL, DELTA
)
typical_scale_prime = typical_scale
TheoryROCVisualizer(
    distribution='Gaussian',
    scale=1401.9596894231056,
    delta=1e-06,
    selectable_distribution=False,
    loc_negative=34063.394,
    scale_negative=1401.9596894231056,
    loc_positive=34395.244,
    scale_positive=1401.9596894231056,
    bound_epsilon=1.0,
    show_governing_point=False,
).embed()


Now move to an engineered micro-example. The pair is still a valid substitution-neighbor pair of size `N=1000`, but the rows are arranged so the 99% quantile sits just below a flat upper tail ($30k bulk → $120k tail). Replacing the **top bulk order statistic** (not a low row) by a witness above the tail jumps the learned upper clip by ~$90k and sharply increases the Gaussian noise scale, while the released-output distributions still overlap enough to audit the failure.


In [ ]:
# Engineered pair: q0.99 sits between the bulk max (~30k) and a flat 120k tail.
(
    fabricated_quantile_D,
    fabricated_quantile_D_prime,
    fabricated_out_idx,
    fabricated_witness,
) = build_engineered_quantile_neighbor_pair(N)
assert len(fabricated_quantile_D) == len(fabricated_quantile_D_prime) == N
q99_D = float(np.quantile(fabricated_quantile_D, HIGH_Q))
q99_D_prime = float(np.quantile(fabricated_quantile_D_prime, HIGH_Q))
print(
    f"learned upper quantile q{HIGH_Q:g}: "
    f"D={q99_D:,.0f}  D'={q99_D_prime:,.0f}  "
    f"shift={q99_D_prime - q99_D:,.0f}"
)


The next plot shows the engineered neighboring pair. Only **one row** differs between `D` and `D'`. The dashed red **q0.99** line jumps from ~$31k to $120k even though the sample means barely move.


In [ ]:
fig = sorted_neighbor_bars_figure(
    fabricated_quantile_D,
    fabricated_quantile_D_prime,
    title='Engineered pair: one substitution moves the learned upper quantile',
    yscale='linear',
    provenance=DataProvenance.CONSTRUCTED_WITNESS,
)
plt.show()


This is the primary privacy-failure ROC for the engineered empirical-quantile pair. It uses the analytical output laws of the broken data-dependent clipping rule: two Gaussians with different deterministic clipped means and different learned noise stds on `D` and `D'`. The thick purple line is the tight `(ε, δ)`-DP bound for this ROC, with the governing point marked where it meets the curve. That tight ε exceeds the claimed `epsilon=1` budget.


In [ ]:
fabricated_mu, fabricated_scale = empirical_quantile_output_law(
    fabricated_quantile_D, LOW_Q, HIGH_Q, EPS_TOTAL, DELTA
)
fabricated_mu_prime, fabricated_scale_prime = empirical_quantile_output_law(
    fabricated_quantile_D_prime, LOW_Q, HIGH_Q, EPS_TOTAL, DELTA
)
TheoryROCVisualizer(
    distribution='Gaussian',
    scale=45.622260337497565,
    delta=1e-06,
    selectable_distribution=False,
    loc_negative=25059.555106167838,
    scale_negative=45.622260337497565,
    loc_positive=26040.555106167845,
    scale_positive=422.0411493765224,
    show_governing_point=True,
    compute_epsilon=True,
    show_compute_epsilon_toggle=False,
).embed()


One accuracy draw for empirical quantile clipping: learn bounds from the sample, clip, calibrate Gaussian noise to the clip width, release the noisy clipped mean. Compare to the sample mean on the same draw.


In [ ]:
_eq_rng = np.random.default_rng(SEED_MAP["accuracy_empirical_quantile"])
_eq_draw = _eq_rng.choice(income_all, size=N, replace=False)
_eq_run_rng = np.random.default_rng(_eq_rng.integers(0, 2**31 - 1))
_eq_L = float(np.quantile(_eq_draw, LOW_Q))
_eq_U = float(np.quantile(_eq_draw, HIGH_Q))
_eq_estimate = empirical_quantile_clipped_mean(
    _eq_draw, EPS_TOTAL, LOW_Q, HIGH_Q, _eq_run_rng
)
_eq_target = float(np.mean(_eq_draw))
print(
    f"L={_eq_L:,.0f}, U={_eq_U:,.0f}, estimate={_eq_estimate:,.0f}, "
    f"target={_eq_target:,.0f}, error={_eq_estimate - _eq_target:,.0f}"
)


Typical-accuracy sweep for the invalid empirical-quantile clipped mean under the fixed contract (`N_DATASETS` replicates × `N_RUNS` mechanism draw each). X-axis: signed error in dollars. Y-axis: probability per bin.


In [ ]:
emp_q_rows = evaluate_accuracy(
    empirical_quantile_clipping_attempt,
    income_all,
    mean_targets,
    n=N,
    n_datasets=N_DATASETS,
    n_runs=N_RUNS,
    seed=SEED_MAP["accuracy_empirical_quantile"],
    method='empirical quantile clipping',
    privacy_status='invalid',
    estimate_target='mean',
    epsilon_total=EPS_TOTAL,
    notes='non-private quantile bounds; Gaussian noise on clipped mean only',
)
leaderboard_parts.append(emp_q_rows)
fig = make_accuracy_leaderboard_figure(emp_q_rows, estimate_target='mean', error_metric='value')
plt.show()


Privacy audit on the engineered quantile-gap pair: run the broken mechanism on `D` and `D'`, select threshold τ★ from the empirical ROC, and read off plug-in ε̂. Left panel: sampled output densities with τ★ marked. Right panel: empirical ROC with governing point.


In [ ]:
S3_AUDIT_TRIALS = N_DATASETS * N_RUNS

def empirical_quantile_audit_mechanism(x, rng):
    return empirical_quantile_clipped_mean(x, EPS_TOTAL, LOW_Q, HIGH_Q, rng)

panel_s3 = audit_panel(
    empirical_quantile_audit_mechanism,
    fabricated_quantile_D,
    fabricated_quantile_D_prime,
    n_trials=S3_AUDIT_TRIALS,
    delta=DELTA,
    seed=SEED_MAP["audit_s3_empirical_quantile"],
    adversary_statistic='released empirical-quantile clipped mean',
)
print(
    f"tau*={panel_s3.tau_star:.3g}, eps_plug={panel_s3.eps_plug:.3g}, "
    f"separates={panel_s3.separates}"
)


Interactive empirical ROC explorer for the S3 audit samples. Toggle **Compute ε** to overlay the claimed `(ε, δ)` bound and the ROC-selected threshold τ★ on the output-density panel.


In [ ]:
PrivateEstimationAuditROCVisualizer(scene="quantile-s3").embed()


**Diagnosis:** adding noise to the final number does not privatize the earlier data-dependent clipping choice.


## S4 — Median as a Robust Statistic

The median looks robust on ordinary Fulton data: a typical substitution neighbor moves it only a little. But value sensitivity is measured by the worst-case neighbor, not the typical one. The engineered pair below arranges rows so one substitution jumps the median across a wide value gap.


Start with a typical size-`N=1000` Fulton substitution pair. The median moves only a little under an ordinary real-data swap, which is why the median looks robust on benign data.


In [ ]:
typical_median_shift, typical_median_out_idx, typical_median_in_value, typical_median_D_prime = select_typical_median_pair(
    D,
    witness_pool,
    max_shift=1_000.0,
    seed=SEED_MAP["typical_median_pair"],
)
assert len(D) == len(typical_median_D_prime) == N
typical_median_trace_rows = median_perturbation_trace(
    D, typical_median_D_prime, typical_median_out_idx
)


The next plot uses the same sorted-neighbor style as the previous failed-estimator examples. Both panels have `N=1000` rows; the horizontal dotted line marks the sample median.


In [ ]:
fig = sorted_neighbor_bars_figure(
    D,
    typical_median_D_prime,
    title=f'Typical median pair: one substitution changes median by {typical_median_shift:,.0f}',
    provenance=DataProvenance.REAL_DATA,
    yscale='linear',
    show_quantiles=False,
    show_median=True,
)
plt.show()


Now show the engineered size-`N=1000` median-gap witness. This is still substitution adjacency, but the rows are arranged so one changed row moves the median across the constructed value gap.


In [ ]:
median_artifact = build_median_section_artifact(seed=SEED)
assert len(median_artifact.D) == len(median_artifact.D_prime) == N
median_witness_rows = neighbor_witness_table(
    section='S4 median value sensitivity',
    d_description=f'engineered median-gap subset (n={len(median_artifact.D)}; 501 low + 499 high before swap)',
    d_prime_description=f'substitute row {median_artifact.out_idx} with high-side row {median_artifact.witness:,.0f}',
    adjacency='substitution (one row out, one row in; n unchanged)',
    n=len(median_artifact.D),
    algorithm_effect='median jumps across the constructed value gap',
    audit_target='released noisy median value',
    provenance=DataProvenance.CONSTRUCTED_WITNESS,
)
median_trace_rows = median_perturbation_trace(
    median_artifact.D, median_artifact.D_prime, median_artifact.out_idx
)
engineered_median_shift = abs(float(np.median(median_artifact.D_prime)) - float(np.median(median_artifact.D)))


The next plot uses the same sorted-neighbor style for the engineered pair. The large vertical gap around the middle ranks is the value-sensitivity problem.


In [ ]:
fig = sorted_neighbor_bars_figure(
    median_artifact.D,
    median_artifact.D_prime,
    title=f'Engineered median-gap pair: one substitution changes median by {engineered_median_shift:,.0f}',
    provenance=DataProvenance.CONSTRUCTED_WITNESS,
    yscale='linear',
    show_quantiles=False,
    show_median=True,
)
plt.show()


Naive proposal: release the sample median plus value noise calibrated to the full public value range. That calibration is privacy-valid, but the noise std dwarfs both the typical and engineered median shifts. The next figure applies the signed-error decomposition in dollars. The three lines are `sample median - full Fulton median`, `released estimate - sample median`, and `released estimate - full Fulton median`.


In [ ]:
MEDIAN_VALUE_PUBLIC_RANGE = float(PUBLIC_INCOME_CANDIDATES[-1] - PUBLIC_INCOME_CANDIDATES[0])
median_value_noise_std = gaussian_noise_std(MEDIAN_VALUE_PUBLIC_RANGE, EPS_TOTAL, DELTA)
median_value_noise_variance = median_value_noise_std**2
median_value_noise_attempt = partial(
    median_value_plus_noise,
    epsilon=EPS_TOTAL,
    public_range=MEDIAN_VALUE_PUBLIC_RANGE,
    delta=DELTA,
)

print(f'Median value sensitivity: public range={MEDIAN_VALUE_PUBLIC_RANGE:,.0f}')
print(f'Gaussian std={median_value_noise_std:,.0f}; variance={median_value_noise_variance:,.0f}')


One accuracy draw for median value+noise: sample a dataset, release the noisy median calibrated to the public value range, compare to the sample median.


In [ ]:
_med_rng = np.random.default_rng(SEED_MAP["accuracy_median_value_noise"])
_med_draw = _med_rng.choice(income_all, size=N, replace=False)
_med_run_rng = np.random.default_rng(_med_rng.integers(0, 2**31 - 1))
_med_estimate = median_value_plus_noise(
    _med_draw, EPS_TOTAL, public_range=MEDIAN_VALUE_PUBLIC_RANGE, delta=DELTA, rng=_med_run_rng
)
_med_target = float(np.median(_med_draw))
print(
    f"single draw: target={_med_target:,.0f}, estimate={_med_estimate:,.0f}, "
    f"error={_med_estimate - _med_target:,.0f}"
)


Typical-accuracy sweep for median value+noise under the fixed contract. X-axis: signed error in dollars. Y-axis: probability per bin.


In [ ]:
median_rows = evaluate_accuracy(
    median_value_noise_attempt,
    income_all,
    median_targets,
    n=N,
    n_datasets=N_DATASETS,
    n_runs=N_RUNS,
    seed=SEED_MAP["accuracy_median_value_noise"],
    method='median value+noise',
    privacy_status='valid but low-utility',
    estimate_target='median',
    epsilon_total=EPS_TOTAL,
    notes='Gaussian noise calibrated to the full public value range',
)
leaderboard_parts.append(median_rows)
fig = make_accuracy_leaderboard_figure(median_rows, estimate_target='median', error_metric='value')
plt.show()


The next figure uses signed CDF error instead of dollars, with rank normalized by reference size. The distribution-error line is `rank(release) / |full data| − 0.5`; the empirical-error line is `rank(release) / 1000 − 0.5`; the gray line is the sample-vs-population CDF shift at the released value. X-axis: signed CDF error (rank / size − 0.5). Y-axis: probability per bin.

**Why it piles up at ±0.5:** the value noise here has std ≈ 657k (calibrated to the full 350k public range, the median's worst-case value sensitivity) around a ≈ 21k median, so the released number lands *below the smallest* or *above the largest* row most of the time. Its rank then saturates at 0 or n, sending the CDF error to −0.5 or +0.5. The bimodal shape is the signature of value-noise destroying the median — and the motivation for the rank-based repair in S5, whose sensitivity is 1 regardless of the value range.

In [ ]:
fig = make_accuracy_leaderboard_figure(median_rows, estimate_target='median', error_metric='rank')
plt.show()


Do not audit the range-calibrated median value-noise mechanism here. It is already privacy-valid because the Gaussian std is calibrated to the full public value range, and that std is much larger than the engineered median jump. An empirical ROC is therefore nearly random and only restates that the mechanism hides the jump by adding unusably large noise. The useful lesson is utility: value noise protects privacy by destroying the released dollar value, so the repair changes the metric to rank error.


## S5 — Repair 1: Median via Inverse Sensitivity

For a value `t`, the score is the negative rank error: `-|#{x_i <= t} - alpha n|`. This is an inverse-sensitivity idea: values close to the desired rank need fewer row changes to become optimal, so they get higher probability. Under substitution, the rank changes by at most one, so the score sensitivity is 1.

We realize this as the **grid-free interval exponential mechanism** over the public range **[0, 10M]**: partition the range at the sorted sample, give every interval between adjacent order statistics its (constant) rank-distance score weighted by `log(width)`, draw one interval by the exponential mechanism, and release a uniform point inside it. The output is continuous — there is **no candidate grid**, so accuracy is not hostage to an arbitrary resolution.

The next plot visualizes the **score landscape** on a public display grid (it is only for plotting; the mechanism itself samples continuously). Left: rank loss for each value `t`. Right: the corresponding exponential-mechanism weight `∝ exp(ε·score/2)` at epsilon = 1. Both x-axes are values in thousands of dollars.

In [ ]:
ALPHA = 0.5
# Display-only score landscape on a public grid; the mechanism itself (below) is grid-free.
score_grid = PUBLIC_INVERSE_SENSITIVITY_THRESHOLDS
D_list = list(D)
utilities = np.array([rank_utility(D_list, float(t), ALPHA) for t in score_grid])
rank_loss = -utilities
weights = np.exp(EPS_TOTAL * utilities / 2.0)  # sensitivity = 1
weight_landscape = weights / weights.sum()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(score_grid / 1000, rank_loss, color='C0')
axes[0].set_xlabel('value t (thousand dollars)')
axes[0].set_ylabel('rank loss')
axes[0].set_title('Inverse-sensitivity score: distance from median rank')
axes[1].plot(score_grid / 1000, weight_landscape, color='C2')
axes[1].set_xlabel('value t (thousand dollars)')
axes[1].set_ylabel('relative exp-mechanism weight')
axes[1].set_title('Exponential-mechanism weight landscape, epsilon = 1')
plt.tight_layout(); plt.show()

# Grid-free release: continuous interval exponential mechanism over public range [0, 10M].
private_median = private_inverse_sensitivity_quantile(
    D, rng,
    lower_bound=PUBLIC_QUANTILE_LOWER,
    upper_bound=PUBLIC_QUANTILE_UPPER,
    alpha=ALPHA,
    epsilon=EPS_TOTAL,
)
exact_median = float(np.median(D))
print(f'Exact sample median: {exact_median:,.0f} | one private interval-exponential draw: {private_median:,.2f}')

private_inverse_sensitivity_median = partial(
    private_inverse_sensitivity_quantile,
    lower_bound=PUBLIC_QUANTILE_LOWER,
    upper_bound=PUBLIC_QUANTILE_UPPER,
    alpha=ALPHA,
    epsilon=EPS_TOTAL,
)

The next figure adds the valid private inverse-sensitivity median to the median value track. It uses the same three signed-error lines in dollar units, so the private repair can be compared with the earlier median value+noise proposal.


One accuracy draw for the private inverse-sensitivity median before the sweep.


In [ ]:
_pq_rng = np.random.default_rng(SEED_MAP["accuracy_private_quantile"])
_pq_draw = _pq_rng.choice(income_all, size=N, replace=False)
_pq_run_rng = np.random.default_rng(_pq_rng.integers(0, 2**31 - 1))
_pq_estimate = private_inverse_sensitivity_quantile(
    _pq_draw,
    _pq_run_rng,
    lower_bound=PUBLIC_QUANTILE_LOWER,
    upper_bound=PUBLIC_QUANTILE_UPPER,
    alpha=ALPHA,
    epsilon=EPS_TOTAL,
)
_pq_target = float(np.median(_pq_draw))
print(
    f"single draw: target={_pq_target:,.0f}, estimate={_pq_estimate:,.0f}, "
    f"error={_pq_estimate - _pq_target:,.0f}"
)


Typical-accuracy sweep for the private inverse-sensitivity median. X-axis: signed error in dollars. Y-axis: probability per bin.


In [ ]:
private_quantile_rows = evaluate_accuracy(
    private_inverse_sensitivity_median,
    income_all,
    median_targets,
    n=N,
    n_datasets=N_DATASETS,
    n_runs=N_RUNS,
    seed=SEED_MAP["accuracy_private_quantile"],
    method='private inverse-sensitivity median',
    privacy_status='valid',
    estimate_target='median',
    epsilon_total=EPS_TOTAL,
    notes='grid-free interval exponential mechanism over public range [0, 10M]',
)
leaderboard_parts.append(private_quantile_rows)
fig = make_accuracy_leaderboard_figure(private_quantile_rows, estimate_target='median', error_metric='value')
plt.show()


The next figure uses signed CDF error instead of dollars, with rank normalized by reference size. The distribution-error line is `rank(release) / |full data| − 0.5`; the empirical-error line is `rank(release) / 1000 − 0.5`; the gray line is the sample-vs-population CDF shift at the released value. X-axis: signed CDF error (rank / size − 0.5). Y-axis: probability per bin.


In [ ]:
fig = make_accuracy_leaderboard_figure(private_quantile_rows, estimate_target='median', error_metric='rank')
plt.show()


## S6 — Private Quantile-Clipped Mean

Now return to the original mean problem: release **L** and **U** with the same grid-free interval exponential mechanism as S5 (rank-distance score over the public range **[0, 10M]**), clip to `[L, U]`, then add Gaussian noise to the clipped mean calibrated to `(U−L)/n`. The composed mechanism tracks its privacy-budget ledger in variables rather than rendering dataframe output.

In [ ]:
EPS_LOW_Q = 0.25
EPS_HIGH_Q = 0.25
EPS_MEAN = 0.50
assert abs((EPS_LOW_Q + EPS_HIGH_Q + EPS_MEAN) - EPS_TOTAL) < 1e-12

LOW_Q, HIGH_Q = 0.01, 0.99

# Grid-free interval exponential-mechanism quantiles, then noisy clipped mean.
L = private_quantile_exponential(D, PUBLIC_QUANTILE_LOWER, PUBLIC_QUANTILE_UPPER, EPS_LOW_Q, rng, quantile=LOW_Q)
U = private_quantile_exponential(D, PUBLIC_QUANTILE_LOWER, PUBLIC_QUANTILE_UPPER, EPS_HIGH_Q, rng, quantile=HIGH_Q)
L, U = enforce_clip_bounds(L, U)
clipped = np.clip(D, L, U)
mean_sensitivity = (U - L) / len(D)
noise_std = gaussian_noise_std(mean_sensitivity, EPS_MEAN, DELTA)
private_estimate = float(np.mean(clipped) + rng.normal(0.0, noise_std))

result = {
    'estimate': private_estimate,
    'L': L,
    'U': U,
    'epsilon': {'low_q': EPS_LOW_Q, 'high_q': EPS_HIGH_Q, 'mean': EPS_MEAN},
    'sensitivity': {'quantile': 1, 'mean': mean_sensitivity},
}
print(f'Private L={L:,.0f}, U={U:,.0f} | (U-L)/n={mean_sensitivity:,.0f} | Gaussian std={noise_std:,.0f}')
print(f'Private estimate: {private_estimate:,.0f} | exact clipped mean on same D: {np.mean(clipped):,.0f}')
private_qcm_budget_rows = budget_ledger_table(result)


def private_quantile_clipped_mean_visible(x, rng):
    L = private_quantile_exponential(x, PUBLIC_QUANTILE_LOWER, PUBLIC_QUANTILE_UPPER, EPS_LOW_Q, rng, quantile=LOW_Q)
    U = private_quantile_exponential(x, PUBLIC_QUANTILE_LOWER, PUBLIC_QUANTILE_UPPER, EPS_HIGH_Q, rng, quantile=HIGH_Q)
    L, U = enforce_clip_bounds(L, U)
    clipped = np.clip(x, L, U)
    mean_sens = (U - L) / len(x)
    noise = gaussian_noise_std(mean_sens, EPS_MEAN, DELTA)
    return {
        'estimate': float(np.mean(clipped) + rng.normal(0.0, noise)),
        'L': L,
        'U': U,
        'epsilon': {'low_q': EPS_LOW_Q, 'high_q': EPS_HIGH_Q, 'mean': EPS_MEAN},
        'sensitivity': {'quantile': 1, 'mean': mean_sens},
    }

The next figure adds the valid private quantile-clipped mean to the mean track. It uses the same three signed-error lines: sampling gap, mechanism error, and total error.


One accuracy draw for private quantile-clipped mean: release private L, U, and noisy clipped mean on the same dataset draw.


In [ ]:
_qcm_rng = np.random.default_rng(SEED_MAP["accuracy_private_qcm"])
_qcm_draw = _qcm_rng.choice(income_all, size=N, replace=False)
_qcm_run_rng = np.random.default_rng(_qcm_rng.integers(0, 2**31 - 1))
_qcm_result = private_quantile_clipped_mean_visible(_qcm_draw, _qcm_run_rng)
_qcm_target = float(np.mean(_qcm_draw))
print(
    f"L={_qcm_result['L']:,.0f}, U={_qcm_result['U']:,.0f}, "
    f"estimate={_qcm_result['estimate']:,.0f}, target={_qcm_target:,.0f}, "
    f"error={_qcm_result['estimate'] - _qcm_target:,.0f}"
)


Typical-accuracy sweep for private quantile-clipped mean. X-axis: signed error in dollars. Y-axis: probability per bin.


In [ ]:
private_qcm_rows = evaluate_accuracy(
    private_quantile_clipped_mean_visible,
    income_all,
    mean_targets,
    n=N,
    n_datasets=N_DATASETS,
    n_runs=N_RUNS,
    seed=SEED_MAP["accuracy_private_qcm"],
    method='private quantile-clipped mean',
    privacy_status='valid',
    estimate_target='mean',
    epsilon_total=EPS_TOTAL,
    notes='grid-free interval-exponential L,U plus noisy clipped mean',
)
leaderboard_parts.append(private_qcm_rows)
fig = make_accuracy_leaderboard_figure(private_qcm_rows, estimate_target='mean', error_metric='value')
plt.show()


## S7 — Noisy Histograms and the Binning Problem

Before the final Gaussian-style estimator, isolate the histogram issue. Exact EDA is forbidden, but a fixed-bin noisy histogram is private if the bins are public. The utility problem is binning: bins that are too wide hide the feature we need, while bins that are too narrow spend the same noisy count vector over many unstable cells.


The next plot is a private fixed-bin histogram of the size-1000 sample `D`. X-axis: public income bins. Y-axis: noisy counts. The bins were fixed before seeing the private sample, and each substitution changes at most two bin counts.


In [ ]:
hist = fixed_bin_noisy_histogram(D, PUBLIC_INCOME_BIN_EDGES, epsilon=EPS_SCALE, rng=rng)
centers = 0.5 * (hist['bin_edges'][:-1] + hist['bin_edges'][1:])
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(centers, hist['counts'], width=centers[1] - centers[0], color=PROVENANCE_STYLE[DataProvenance.PRIVATE_RUN].color, alpha=0.75)
ax.set_xlabel('income bin center ($)')
ax.set_ylabel('noisy count')
ax.set_title(f'Private fixed-bin noisy histogram of sampled D (n={len(D):,}, epsilon={EPS_SCALE})')
plt.show()


This cell computes the histogram sensitivity trace without rendering dataframe output: one substitution leaves one bin and enters another, so at most two bin counts change.


In [ ]:
out_candidates = np.where((D >= 10_000) & (D < 20_000))[0]
out_i = int(out_candidates[0]) if len(out_candidates) else int(np.argmin(D))
histogram_trace_rows = histogram_replacement_trace(
    D, PUBLIC_INCOME_BIN_EDGES, out_i, 345_000.0
)


The sensitivity trace gives the sensitivity story, but not the accuracy story. Even with the right noise calibration, a fixed income grid can be a poor scale-localization tool when the relevant spread is multiplicative or when the useful bin width depends on the unknown standard deviation.


## S8 — Gaussian Localization via Pairwise Differences (KV18)

The final scale-based idea follows Karwa & Vadhan (2018, [arXiv:1711.03908](https://arxiv.org/pdf/1711.03908)). The key trick: take **disjoint pairwise absolute differences** `|X_i − X_j|`. Each difference is the absolute value of an `N(0, 2σ²)` variable, so the **unknown mean cancels** and the differences depend only on the scale `σ`. A noisy histogram of those differences localizes `σ` (each row sits in one pair, so the count vector has bounded sensitivity); the selected scale then sets the resolution for a private location search and the final `μ ± 3σ` clipping range used for the mean.

This is a **model-dependent** estimator: privacy holds by composition, but accuracy depends on the Gaussian assumption. We therefore validate it twice — first on **synthetic Gaussian** data, where it is accurate, then on **Fulton income**, where the heavy tail breaks the assumption and the mean estimate is biased low.

In [ ]:
# KV18 capstone: localize scale from a noisy histogram of disjoint pairwise
# |differences|, then location, then the clipped noisy mean. All grids are public.
gaussian_histogram_mean_visible = lambda x, rng: gaussian_histogram_mean(
    x,
    EPS_SCALE,
    EPS_LOCATION,
    EPS_GAUSS_MEAN,
    PUBLIC_SCALE_DIFF_BIN_EDGES,
    PUBLIC_INCOME_BIN_EDGES,
    rng,
    DELTA,
)

scale_demo = private_scale_from_pairwise_diffs(D, EPS_SCALE, PUBLIC_SCALE_DIFF_BIN_EDGES, rng)
hist = scale_demo['histogram']
centers = 0.5 * (hist['bin_edges'][:-1] + hist['bin_edges'][1:])
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(centers, hist['counts'], width=np.diff(hist['bin_edges']), align='center', color=PROVENANCE_STYLE[DataProvenance.PRIVATE_RUN].color, alpha=0.75)
ax.axvline(scale_demo['median_diff'], color='black', linestyle='--', linewidth=1, label=f'noisy median |diff| {scale_demo["median_diff"]:,.0f}')
ax.set_xlabel('pairwise |difference| bin center ($)')
ax.set_ylabel('noisy count')
ax.set_title(f'Private histogram of disjoint pairwise |differences| (sigma_tilde={scale_demo["sigma_tilde"]:,.0f})')
ax.legend(fontsize=8)
plt.show()

The selected scale `sigma_tilde` now determines the resolution for the private location search and the final `mu ± 3 sigma` clipping range. The next computation runs the full pipeline (scale from pairwise differences → location → clipped noisy mean) and records the intermediate private quantities.

In [ ]:
kv_demo = gaussian_histogram_mean_visible(D, rng)
kv_demo_summary = {
    'private_sigma': kv_demo['sigma_tilde'],
    'private_mu_bin': kv_demo['mu_hat'],
    'clip_L': kv_demo['L'],
    'clip_U': kv_demo['U'],
    'private_estimate': kv_demo['estimate'],
    'sample_mean': float(np.mean(D)),
}

The next plot uses the same private sigma-bin output to draw the private location histogram and the final clipping interval.


In [ ]:
loc_hist = kv_demo['location']['histogram']
loc_centers = 0.5 * (loc_hist['bin_edges'][:-1] + loc_hist['bin_edges'][1:])
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(loc_centers, loc_hist['counts'], width=np.diff(loc_hist['bin_edges']), color='C2', alpha=0.75)
ax.axvline(kv_demo['mu_hat'], color='black', linestyle='--', linewidth=1, label=f'selected location {kv_demo["mu_hat"]:,.0f}')
ax.axvspan(kv_demo['L'], kv_demo['U'], color='C2', alpha=0.12, label='final clipping range')
ax.set_xlabel('income bin center')
ax.set_ylabel('noisy count')
ax.set_title('Private location histogram with bin width determined by private sigma')
ax.legend(fontsize=8)
plt.show()


Before income, validate where the assumption holds: a **synthetic Gaussian** population with a known mean. Here scale and location are well-defined, so the pairwise-difference σ̃, the location histogram μ̂, and the `μ̂ ± 3σ̃` clip all behave and the estimate tracks the true mean. The next figure is the signed-error decomposition on that synthetic population (mechanism error should be small and centered).

In [ ]:
GAUSS_MU, GAUSS_SIGMA = 40_000.0, 12_000.0
gaussian_population = np.random.default_rng(SEED_MAP["gaussian_synth_population"]).normal(
    GAUSS_MU, GAUSS_SIGMA, size=25_000
)


One accuracy draw on synthetic Gaussian data before the full sweep.


In [ ]:
_gs_rng = np.random.default_rng(SEED_MAP["accuracy_gaussian_loc_synth"])
_gs_draw = _gs_rng.choice(gaussian_population, size=N, replace=False)
_gs_run_rng = np.random.default_rng(_gs_rng.integers(0, 2**31 - 1))
_gs_result = gaussian_histogram_mean_visible(_gs_draw, _gs_run_rng)
_gs_target = float(np.mean(_gs_draw))
print(
    f"single draw: target={_gs_target:,.0f}, estimate={_gs_result['estimate']:,.0f}, "
    f"error={_gs_result['estimate'] - _gs_target:,.0f}"
)


Full accuracy sweep on synthetic Gaussian data. X-axis: signed error in dollars. Y-axis: probability per bin.


In [ ]:
# Side validation on synthetic Gaussian data (not appended to the income leaderboard).
gaussian_loc_synth_rows = evaluate_accuracy(
    gaussian_histogram_mean_visible,
    gaussian_population,
    mean_targets,
    n=N,
    n_datasets=N_DATASETS,
    n_runs=N_RUNS,
    seed=SEED_MAP["accuracy_gaussian_loc_synth"],
    method='gaussian localization mean — synthetic Gaussian',
    privacy_status='model_dependent',
    estimate_target='mean',
    epsilon_total=EPS_TOTAL,
    notes='KV18 pairwise-difference localization on data that is actually Gaussian',
)
fig = make_accuracy_leaderboard_figure(gaussian_loc_synth_rows, estimate_target='mean', error_metric='value')
plt.show()


Now the **same mechanism on Fulton income**, which is heavy-tailed rather than Gaussian. The model breaks at every step: the pairwise-difference scale sees only the bulk spread (σ̃ ≈ 20k vs a true std ≈ 62k), the location histogram locks onto the low-income **mode** (μ̂ ≈ 6k, well below the 39k mean), and the resulting `μ̂ ± 3σ̃` clip discards the upper tail that drives the mean. The estimate is therefore biased **low** (≈ 27k vs 39k). Privacy is still valid by composition; only **accuracy** fails — exactly because the Gaussian assumption does not hold. Compare the centering/spread here with the synthetic-Gaussian panel above.

One accuracy draw on Fulton income before the full Gaussian-localization sweep.


In [ ]:
_gf_rng = np.random.default_rng(SEED_MAP["accuracy_gaussian_loc_fulton"])
_gf_draw = _gf_rng.choice(income_all, size=N, replace=False)
_gf_run_rng = np.random.default_rng(_gf_rng.integers(0, 2**31 - 1))
_gf_result = gaussian_histogram_mean_visible(_gf_draw, _gf_run_rng)
_gf_target = float(np.mean(_gf_draw))
print(
    f"single draw: target={_gf_target:,.0f}, estimate={_gf_result['estimate']:,.0f}, "
    f"error={_gf_result['estimate'] - _gf_target:,.0f}"
)


Full accuracy sweep on Fulton income with the Gaussian-localization mechanism. X-axis: signed error in dollars. Y-axis: probability per bin.


In [ ]:
gaussian_loc_rows = evaluate_accuracy(
    gaussian_histogram_mean_visible,
    income_all,
    mean_targets,
    n=N,
    n_datasets=N_DATASETS,
    n_runs=N_RUNS,
    seed=SEED_MAP["accuracy_gaussian_loc_fulton"],
    method='gaussian localization mean — Fulton income',
    privacy_status='model_dependent',
    estimate_target='mean',
    epsilon_total=EPS_TOTAL,
    notes='KV18 pairwise-difference scale, then location and clipped mean; Gaussian assumption fails on heavy-tailed income',
)
leaderboard_parts.append(gaussian_loc_rows)
fig = make_accuracy_leaderboard_figure(gaussian_loc_rows, estimate_target='mean', error_metric='value')
plt.show()


## S9 — Summary and Transition

The summary artifacts are generated from the leaderboard for downstream use, but the notebook mainline does not render dataframe outputs.


In [ ]:
leaderboard = pd.concat(leaderboard_parts, ignore_index=True) if leaderboard_parts else pd.DataFrame()
accuracy_summary = accuracy_summary_table(leaderboard)
proposal_summary = pd.DataFrame(proposals_diagnoses_repairs_from_leaderboard(leaderboard))
